In [1]:
from scipy.stats import randint, uniform, loguniform
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier
)


models = {
            "Random Forest": RandomForestClassifier(
                verbose=1,
                n_jobs=-1,
                random_state=42
            ),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
            "Gradient Boosting": GradientBoostingClassifier(
                verbose=1,
                random_state=42
            ),
            "Logistic Regression": LogisticRegression(
                max_iter=1000,
                random_state=42
            ),
            "AdaBoost": AdaBoostClassifier(random_state=42),
        }

params = {
    "Decision Tree": {
        "criterion": ["gini", "entropy", "log_loss"],
        "splitter": ["best", "random"],
        "max_depth": randint(3, 30),
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", None],
    },
    "Random Forest": {
        "n_estimators": randint(50, 500),
        "max_depth": [None] + list(range(5, 31, 5)),
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", None],
    },
    "Gradient Boosting": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(1e-3, 3e-1),
        "subsample": uniform(0.6, 0.4),  # U[0.6, 1.0]
        "max_depth": randint(2, 8),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", None],
    },
    "Logistic Regression": {
        "C": loguniform(1e-3, 1e2),
    },
    "AdaBoost": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(1e-3, 1),
    },
}


In [2]:
model = list(models.values())[0]
print(model)

RandomForestClassifier(n_jobs=-1, random_state=42, verbose=1)


In [3]:
param = list(params.keys())
params[param[0]]

{'criterion': ['gini', 'entropy', 'log_loss'],
 'splitter': ['best', 'random'],
 'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7f3131d981a0>,
 'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7f3131f0bed0>,
 'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7f3131d9c190>,
 'max_features': ['sqrt', 'log2', None]}

In [4]:
for key in models.keys():
    model = models[key]
    param = params[key]
print(model)
print(param)

AdaBoostClassifier(random_state=42)
{'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f3131f8af50>, 'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f3131f23e10>}


## Random search CV
Tune each model with `RandomizedSearchCV`: it samples `n_iter` random combinations from `params` instead of trying the full grid.

In [5]:
import numpy as np
from pathlib import Path

# latest transformed artifacts; last column = target
latest = sorted(Path("../Artifacts").glob("*/data_transformation/transformed"))[-1]
train_arr = np.load(latest / "train.npy")
test_arr = np.load(latest / "test.npy")

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]
print(latest, X_train.shape, X_test.shape)

../Artifacts/11_18_2026_16_39_53/data_transformation/transformed (8844, 30) (2211, 30)


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score

# quiet the verbose estimators for the search
for m in models.values():
    if hasattr(m, "set_params") and "verbose" in m.get_params():
        m.set_params(verbose=0)

results = {}
for name, model in models.items():
    grid = params[name]
    if not grid:  # nothing to tune (e.g. Logistic Regression)
        model.fit(X_train, y_train)
        results[name] = {"cv_f1": None, "test_f1": f1_score(y_test, model.predict(X_test)), "best_params": {}}
        continue

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=grid,
        n_iter=10,          # random combos sampled per model
        cv=3,
        scoring="f1",
        n_jobs=-1,
        random_state=42,
    )
    search.fit(X_train, y_train)
    results[name] = {
        "cv_f1": search.best_score_,
        "test_f1": f1_score(y_test, search.best_estimator_.predict(X_test)),
        "best_params": search.best_params_,
    }

for name, r in results.items():
    print(name, r)

Random Forest {'cv_f1': np.float64(0.9601413944610657), 'test_f1': 0.965991902834008, 'best_params': {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 13, 'n_estimators': 363}}
Decision Tree {'cv_f1': np.float64(0.9472876527142399), 'test_f1': 0.9498355263157895, 'best_params': {'criterion': 'log_loss', 'max_depth': 25, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 5, 'splitter': 'random'}}
Gradient Boosting {'cv_f1': np.float64(0.97109202001557), 'test_f1': 0.9764035801464606, 'best_params': {'learning_rate': np.float64(0.05675206026988748), 'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 393, 'subsample': np.float64(0.9329770563201687)}}
Logistic Regression {'cv_f1': np.float64(0.9360776658054326), 'test_f1': 0.9426494345718901, 'best_params': {'C': np.float64(0.0745934328572655)}}
AdaBoost {'cv_f1': np.float64(0.9361552418386081), 'test_f1': 0.9414125200642055, 'best_params': {'learning_rate': np.

In [7]:
import pandas as pd
pd.DataFrame(results).T.sort_values("test_f1", ascending=False)

,cv_f1,test_f1,best_params
Gradient Boosting,0.971092,0.976404,"{'learning_rate': 0.05675206026988748, 'max_de..."
Random Forest,0.960141,0.965992,"{'max_depth': 15, 'max_features': 'sqrt', 'min..."
Decision Tree,0.947288,0.949836,"{'criterion': 'log_loss', 'max_depth': 25, 'ma..."
Logistic Regression,0.936078,0.942649,{'C': 0.0745934328572655}
AdaBoost,0.936155,0.941413,"{'learning_rate': 0.31428808908401085, 'n_esti..."
